In [1]:
import cvxpy as cp
import numpy as np
import itertools
import random

In [2]:
# -----------------------------
# 2048 engine (same as before)
# -----------------------------
def _compress(row):
    non_zero = row[row != 0]
    return np.concatenate([non_zero, np.zeros(len(row) - len(non_zero), dtype=row.dtype)])

def _merge(row):
    row = row.copy()
    score = 0
    for i in range(len(row) - 1):
        if row[i] != 0 and row[i] == row[i + 1]:
            row[i] *= 2
            row[i + 1] = 0
            score += row[i]
    return row, score

def _move_left(board):
    new_board = np.zeros_like(board)
    total_score = 0
    for i in range(board.shape[0]):
        row = board[i, :]
        row = _compress(row)
        row, score = _merge(row)
        row = _compress(row)
        new_board[i, :] = row
        total_score += score
    return new_board, total_score

def _move_right(board):
    flipped = np.fliplr(board)
    moved, score = _move_left(flipped)
    return np.fliplr(moved), score

def _move_up(board):
    rotated = board.T
    moved, score = _move_left(rotated)
    return moved.T, score

def _move_down(board):
    rotated = board.T
    moved, score = _move_right(rotated)
    return moved.T, score

def apply_move(board, direction):
    if direction == 0: return _move_left(board)
    if direction == 1: return _move_right(board)
    if direction == 2: return _move_up(board)
    if direction == 3: return _move_down(board)
    raise ValueError("Invalid direction")

# -----------------------------
# spawn logic
# -----------------------------
def spawn_tile(board):
    empty = list(zip(*np.where(board == 0)))
    if not empty:
        return board
    i, j = random.choice(empty)
    board[i, j] = 2 if random.random() < 0.9 else 4
    return board

# -----------------------------
# stochastic rollout for a sequence
# -----------------------------
def simulate_sequence_stochastic(board, seq, n_scenarios=20):
    """
    For a fixed move sequence `seq`, simulate n_scenarios random
    spawn trajectories and return the average cumulative score.
    """
    scores = []
    for _ in range(n_scenarios):
        b = board.copy()
        total = 0
        feasible = True
        for d in seq:
            new_b, s = apply_move(b, d)
            if np.array_equal(new_b, b):  # illegal move
                feasible = False
                break
            total += s
            b = spawn_tile(new_b)
        if feasible:
            scores.append(total)
        else:
            scores.append(-1e6)  # heavily penalize infeasible sequences
    return np.mean(scores)

# -----------------------------
# N-stage expected value MIP
# -----------------------------
def mip_n_stage_expected_move(board, N=3, n_scenarios=20):
    D = 4
    sequences = list(itertools.product(range(D), repeat=N))
    K = len(sequences)

    exp_scores = []
    for seq in sequences:
        exp_scores.append(simulate_sequence_stochastic(board, seq, n_scenarios=n_scenarios))
    exp_scores = np.array(exp_scores, dtype=float)

    y = cp.Variable(K, boolean=True)
    constraints = [cp.sum(y) == 1]
    objective = cp.Maximize(exp_scores @ y)
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.HIGHS)

    best_idx = int(np.argmax(y.value))
    best_seq = sequences[best_idx]
    best_score = exp_scores[best_idx]
    return best_seq[0], best_seq, best_score

# -----------------------------
# full game with N-stage EV planner
# -----------------------------
def run_n_stage_ev_game(N=3, n_scenarios=20, seed=None):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    board = np.zeros((4,4), dtype=int)
    board = spawn_tile(board)
    board = spawn_tile(board)

    total_score = 0
    moves = 0

    while True:
        d, seq, ev_score = mip_n_stage_expected_move(board, N=N, n_scenarios=n_scenarios)
        new_board, gained = apply_move(board, d)
        if np.array_equal(new_board, board):
            break
        total_score += gained
        moves += 1
        board = spawn_tile(new_board)

    return {
        "score": total_score,
        "max_tile": int(board.max()),
        "moves": moves,
        "final_board": board
    }

def simulate_n_stage_ev(N=3, n_scenarios=20, n_games=10):
    results = [run_n_stage_ev_game(N=N, n_scenarios=n_scenarios) for _ in range(n_games)]
    scores = [r["score"] for r in results]
    max_tiles = [r["max_tile"] for r in results]

    print(f"N-stage EV planner: N={N}, scenarios={n_scenarios}")
    print(f"Games: {n_games}")
    print(f"Avg score: {np.mean(scores):.1f}")
    print(f"Median score: {np.median(scores):.1f}")
    print(f"Max score: {np.max(scores)}")
    print(f"Avg max tile: {np.mean(max_tiles):.1f}")
    print("Max tile distribution:")
    for t in sorted(set(max_tiles)):
        print(f"  {t}: {max_tiles.count(t)} games")

    return results

# example: #-step EV planner, # scenarios per sequence, # games
simulate_n_stage_ev(N=3, n_scenarios=10, n_games=1)


N-stage EV planner: N=3, scenarios=10
Games: 1
Avg score: 27332.0
Median score: 27332.0
Max score: 27332
Avg max tile: 2048.0
Max tile distribution:
  2048: 1 games


[{'score': np.int64(27332),
  'max_tile': 2048,
  'moves': 1431,
  'final_board': array([[ 512,   16,   32,    2],
         [  64,   32,  128,    4],
         [   8,  256, 2048,   16],
         [   2,    4,   16,    8]])}]

# N‑Stage Expected Value MIP Planner for 2048

This planner extends the one‑move greedy MIP into a **multi‑step lookahead optimizer** that evaluates entire trajectories of length \(N\).  
The key idea is to combine:

1. **Exact 2048 physics** (handled by the Python engine)
2. **Monte‑Carlo sampling of tile spawns**
3. **Trajectory‑level expected value estimation**
4. **A MIP that selects the best trajectory**
5. **Model Predictive Control (MPC)**: apply only the first move

This produces a powerful planning agent that optimizes *expected* cumulative score over multiple future moves.

---

## 1. Trajectory Enumeration

For a planning horizon of length \(N\), we enumerate all possible move sequences:



\$
\mathcal{S} = \{0,1,2,3\}^N
\$



Each sequence  


\$
s = (d_1, d_2, \dots, d_N)
\$


represents a full hypothetical future trajectory.

The number of trajectories is:



\$
|\mathcal{S}| = 4^N
\$



For \(N = 3\), this is 64 sequences — easily manageable.

---

## 2. Stochastic Rollouts for Expected Value

For each sequence \(s\), we simulate multiple random tile‑spawn scenarios:



\$
\hat{V}(s) = \frac{1}{K} \sum_{k=1}^K \text{Score}(s \mid \text{scenario } k)
\$



Each scenario:

1. Applies the move
2. Computes the true merge score
3. Spawns a random tile (2 with 90% probability, 4 with 10%)
4. Continues until the sequence ends or becomes illegal

This produces an **unbiased Monte‑Carlo estimate** of the expected cumulative score for that trajectory.

---

## 3. MIP Over Trajectories

Let \(y_s \in \{0,1\}\) be a binary variable indicating whether trajectory \(s\) is chosen.

### Constraints

Exactly one trajectory is selected:



\$
\sum_{s \in \mathcal{S}} y_s = 1
\$



### Objective

Maximize expected cumulative score:



\$
\max_{y} \quad \sum_{s \in \mathcal{S}} y_s \, \hat{V}(s)
\$



This is a **purely linear** objective because all expected values \(\hat{V}(s)\) are constants.

---

## 4. MPC Execution

Although the MIP chooses an entire trajectory, we only execute the **first move**:



\$
a^\* = s^\*_1
\$



Then:

1. Apply the move
2. Spawn a tile
3. Recompute the N‑stage MIP from the new board
4. Repeat until no legal moves remain

This is standard **Model Predictive Control (MPC)**.

---

## 5. Advantages of This Approach

- **Exact physics**: all transitions use the real 2048 engine  
- **Stochastic modeling**: tile spawns are sampled explicitly  
- **Linear MIP**: no bilinear terms, fully DCP‑compliant  
- **Flexible horizon**: \(N = 2,3,4\) are practical  
- **Strong performance**: significantly better than greedy  

---

## 6. Summary

The N‑stage Expected Value MIP Planner:

- Enumerates all move sequences of length \(N\)
- Estimates expected cumulative score via Monte‑Carlo rollouts
- Uses a MIP to select the best trajectory
- Executes only the first move (MPC)
- Repeats until the game ends

This produces a principled, optimization‑driven 2048 agent that balances immediate reward with multi‑step planning under uncertainty.
